In [2]:
# Cell 1 — imports
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import numpy as np
from dotenv import load_dotenv
load_dotenv('../../.env')

from src.utils.config import settings
from src.utils.logger import get_logger
log = get_logger("setup_check")

log.info("✅ Imports working")

# Cell 2 — load all datasets
files = {
    'movies_metadata': '../../data/raw/movies_metadata.csv',
    'credits':         '../../data/raw/credits.csv',
    'keywords':        '../../data/raw/keywords.csv',
    'links':           '../../data/raw/links.csv',
    'links_small':     '../../data/raw/links_small.csv',
    'ratings_small':   '../../data/raw/ratings_small.csv',
     'ratings':        '../../data/raw/ratings.csv',
}

dfs = {}
for name, path in files.items():
    df = pd.read_csv(path, low_memory=False)
    dfs[name] = df
    print(f"✅ {name:25s} {df.shape[0]:>8,} rows × {df.shape[1]:>3} cols")

print("\n📊 Dataset summary:")
print(f"   Total movies          : {dfs['movies_metadata'].shape[0]:>10,}")
print(f"   Total credits         : {dfs['credits'].shape[0]:>10,}")
print(f"   Total keywords        : {dfs['keywords'].shape[0]:>10,}")
print(f"   Total links           : {dfs['links'].shape[0]:>10,}")
print(f"   Ratings (small/dev)   : {dfs['ratings_small'].shape[0]:>10,}")
print(f"   Ratings (full/prod)   : {dfs['ratings'].shape[0]:>10,}")
print(f"\n   Unique users (full)   : {dfs['ratings']['userId'].nunique():>10,}")
print(f"   Unique movies (full)  : {dfs['ratings']['movieId'].nunique():>10,}")
print(f"   Rating range          : {dfs['ratings']['rating'].min()} – {dfs['ratings']['rating'].max()}")

# Cell 3 — check services
import redis, psycopg2
from qdrant_client import QdrantClient
import mlflow

# Redis
r = redis.Redis(host=settings.REDIS_HOST, port=settings.REDIS_PORT)
print(f"Redis:    {'✅ connected' if r.ping() else '❌ failed'}")

# Qdrant
qc = QdrantClient(host=settings.QDRANT_HOST, port=settings.QDRANT_PORT)
print(f"Qdrant:   ✅ connected — {len(qc.get_collections().collections)} collections")

# MLflow
mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)
exp = mlflow.set_experiment(settings.MLFLOW_EXPERIMENT_NAME)
print(f"MLflow:   ✅ experiment '{exp.name}' ready")

# PostgreSQL
try:
    conn = psycopg2.connect(
        host=settings.POSTGRES_HOST, port=settings.POSTGRES_PORT,
        dbname=settings.POSTGRES_DB, user=settings.POSTGRES_USER,
        password=settings.POSTGRES_PASSWORD
    )
    print("Postgres: ✅ connected")
    conn.close()
except Exception as e:
    print(f"Postgres: ❌ {e}")

2026-06-02 15:23:43 | INFO | setup_check | ✅ Imports working
✅ movies_metadata             45,466 rows ×  24 cols
✅ credits                     45,476 rows ×   3 cols
✅ keywords                    46,419 rows ×   2 cols
✅ links                       45,843 rows ×   3 cols
✅ links_small                  9,125 rows ×   3 cols
✅ ratings_small              100,004 rows ×   4 cols
✅ ratings                   26,024,289 rows ×   4 cols

📊 Dataset summary:
   Total movies          :     45,466
   Total credits         :     45,476
   Total keywords        :     46,419
   Total links           :     45,843
   Ratings (small/dev)   :    100,004
   Ratings (full/prod)   : 26,024,289

   Unique users (full)   :    270,896
   Unique movies (full)  :     45,115
   Rating range          : 0.5 – 5.0


ConnectionError: Error 61 connecting to localhost:6379. Connection refused.